<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# RR Skillverse — Module 2
# Deep Neural Networks (with Adversarial Cybersecurity)

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 2 of 12 · 3 hours · Continues the RR Finance running system built in Module 1*

---

> **Free Learning Handbook — Notebook Edition.** This notebook is a personal educational reference for structured learning and hands-on practice. It is shared for learning purposes and is not presented as a commercial product.

**Format note:** Module 1 was delivered as a browser handbook with VS Code demo scripts. Module 2 is delivered as a **Jupyter notebook** because deep learning work benefits from running one cell at a time, inspecting tensors immediately, and re-running a single experiment without restarting a whole script — which is exactly how you will work in Modules 2–9 of this program.

## The hand-off from Module 1

At the end of Module 1, RR Finance has:

| Artifact | What it is |
|---|---|
| `rr_finance_module1_dataset.csv` | The trusted, inspected loan-risk dataset |
| A logistic regression pipeline | The first default-prediction baseline |
| An Isolation Forest screen | A layer that flags unusual training records |
| A documented poisoning experiment | Evidence that training data is an attack surface |

Module 2 does **not** throw this away. We extend the same RR Finance story in two directions:

1. **Tabular direction (continuity):** we rebuild the default classifier as a small **Artificial Neural Network (ANN)** instead of logistic regression, so you can directly compare "one linear layer + sigmoid" (Module 1) against "several stacked layers" (Module 2) on the *same* problem.
2. **New modality (progression):** RR Finance also receives **scanned cheque images** with handwritten digit amounts that need automated reading. This is a genuine, common financial use case (cheque truncation / MICR-adjacent digit recognition), and it is the natural reason to introduce **Convolutional Neural Networks (CNNs)** — a plain ANN does not scale to images, and this module explains exactly why.

Both threads converge on the same question this module keeps asking: **once a network can learn a decision boundary, can an attacker also learn how to fool it?** That is why adversarial examples (FGSM/PGD) close out the module, in the same "build it, then attack it, then defend it" rhythm as Module 1's poisoning lesson.

## How every lesson is taught (same 6 questions as Module 1)

1. **What problem are we solving?**
2. **Why does it matter in finance?**
3. **Why this technique — what alternatives exist?**
4. **What do the numbers/parameters actually mean?**
5. **What is happening mathematically?**
6. **What happens if we change it?**

Read the markdown, run the code cell immediately below it, inspect the printed output/plot, then move to the next lesson. Do not skip ahead — Lesson 4 (backprop) only makes sense once Lessons 1–3 are solid, and Lesson 9 (CNNs) only makes sense once Lesson 8 (why a plain ANN fails on images) is solid.


## Setup — run this cell first (it is a REAL, runnable cell, not just instructions)

Same fix as Module 1: the cell below is an actual executable `%pip install` cell, not a markdown code block you have to copy elsewhere. It installs straight into this notebook's kernel and is safe to leave in every time you `Run All` — already-installed packages are skipped in a couple of seconds.

**Why PyTorch (and not only TensorFlow)?** The TOC lists both. This notebook standardises on **PyTorch** because its eager-execution style matches how you already think in Jupyter — you run a line, you see a tensor, you run the next line. TensorFlow/Keras equivalents are structurally identical (the six questions do not change); only the API surface differs. If your organisation standardises on TensorFlow, translate cell-by-cell — the math and the teaching narrative stay the same.

**GPU note:** everything in this notebook is intentionally small enough to train on CPU in seconds to a few minutes, so a GPU is a bonus, not a requirement, for Module 2 (Module 4 onward will need one for LLM fine-tuning).

**If you prefer a one-time terminal setup instead:**
```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
pip install torch torchvision matplotlib numpy pandas scikit-learn joblib jupyter
```


In [ ]:
%pip install -q torch torchvision matplotlib numpy pandas scikit-learn joblib
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Foundation first: why neural networks, and why now?

Module 1 built a logistic regression classifier: one linear combination of features, squashed through a sigmoid. It worked — ROC-AUC was respectable — because the relationship between debt-to-income, credit score and default was *roughly* linear-in-the-log-odds.

**The business problem that motivates Module 2:** some patterns are not linear, and some inputs (images, audio, raw text) do not arrive as a clean table of ratios at all. A single straight-line decision boundary cannot represent "risk is high when DTI is high *and* credit history is short, but only when loan-to-income is *also* above a threshold" — that is an interaction, and logistic regression on raw features cannot bend to capture it without you manually engineering the interaction term first.

### Analogy 1 — The single interviewer vs. the panel

Logistic regression is one interviewer with one fixed checklist, scoring an applicant on a weighted sum of answers. A neural network is a panel: several "hidden" interviewers each specialise in a different combination of signals (one focuses on income-vs-debt patterns, another on credit-history-vs-loan-size patterns), and a final decision-maker combines their opinions. Depth (more layers) lets the panel reason about combinations-of-combinations.

### Analogy 2 — Photography and edges

A cheque image is not "16 financial ratios." It is a grid of pixel intensities where meaning lives in *local spatial patterns* — a stroke, a curve, a loop that makes a "6" different from an "8." A plain ANN flattens the image into one long vector and treats pixel (0,0) as no more related to pixel (0,1) than to pixel (27,27). A **CNN** deliberately keeps neighbouring pixels together and slides small pattern-detectors ("filters") across the image — much closer to how a human eye scans a page.

### Analogy 3 — Muscle memory vs. one-shot advice

Gradient descent does not "solve" the network in one step the way you might solve `dti = debt/income` directly. It nudges every weight a small amount, checks whether the error got smaller, and repeats — thousands of times. It is closer to muscle memory built through repetition than to a formula you plug numbers into once.

### Why Module 2 is the right place for cybersecurity's second lesson

Module 1 showed you could poison the **data** a model learns from. Module 2 shows something different and, for many people, more surprising: even a *cleanly trained* network can be fooled at **inference time** by a tiny, human-imperceptible change to the input. That is the adversarial-examples lesson (FGSM/PGD) at the end of this notebook — and it is the reason financial institutions cannot treat "the model works" as the end of the security conversation.


---
## Lesson 1 — The perceptron and linear separability

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | The smallest possible unit of a neural network: can one neuron learn a yes/no rule from examples, instead of being programmed with an if/else rule? |
| **2. Why does it matter in finance?** | Every neuron in every layer of every network in this course — from a 3-layer ANN to a transformer in Module 4 — is a descendant of this same idea: weighted sum → threshold/activation. If you understand this cell, you understand the atomic unit of everything that follows. |
| **3. Why this technique?** | The perceptron learning rule is the simplest possible training algorithm: for each mistake, nudge the weights toward the correct answer. There is no calculus yet — that arrives in Lesson 4. |
| **4. What do the parameters mean?** | `w` (weights) says how much each input matters; `b` (bias) shifts the decision threshold — it is *not* a “bug”, it is what lets the boundary avoid being forced through the origin. `lr` (learning rate) controls how large each correction step is. |
| **5. What is happening mathematically?** | `z = w·x + b`; predict 1 if `z ≥ 0` else 0. On a mistake, `w ← w + lr·(y − ŷ)·x` and `b ← b + lr·(y − ŷ)`. |
| **6. What happens if we change it?** | A single perceptron can only draw a **straight line (or hyperplane)** through the input space. Some patterns simply cannot be separated by *any* straight line — see the XOR example below. That limitation is exactly why we stack neurons into layers. |

### The famous limitation: XOR

AND and OR are linearly separable (one straight line splits the classes). XOR is not — no single straight line can separate `(0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0`. This single-neuron failure, discovered in the 1960s, is historically *why* the field moved to multi-layer networks — which is exactly where Lesson 2 goes next.


In [ ]:
class Perceptron:
    """Smallest possible neural unit: weighted sum + threshold, trained by the perceptron rule."""
    def __init__(self, n_inputs, lr=0.1):
        self.w = np.zeros(n_inputs)
        self.b = 0.0
        self.lr = lr

    def predict(self, x):
        z = np.dot(x, self.w) + self.b
        return 1 if z >= 0 else 0

    def fit(self, X, y, epochs=20):
        for _ in range(epochs):
            for xi, yi in zip(X, y):
                pred = self.predict(xi)
                error = yi - pred
                self.w += self.lr * error * xi
                self.b += self.lr * error

# --- AND gate: linearly separable ---
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0,0,0,1])

p_and = Perceptron(n_inputs=2)
p_and.fit(X_and, y_and)
preds_and = [p_and.predict(x) for x in X_and]
print("AND gate  -> learned:", preds_and, " expected:", list(y_and))

# --- XOR gate: NOT linearly separable ---
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0,1,1,0])

p_xor = Perceptron(n_inputs=2)
p_xor.fit(X_xor, y_xor, epochs=50)
preds_xor = [p_xor.predict(x) for x in X_xor]
print("XOR gate  -> learned:", preds_xor, " expected:", list(y_xor))
print("Notice XOR is NOT fully learned -- this is the historical motivation for multi-layer networks.")


> **Trainer question:** Ask participants to sketch the AND-gate points on paper and draw *any* single straight line separating the 1 from the 0s. Then ask them to try the same for XOR. They will not be able to — that hands-on failure is more convincing than any slide.


---
## Lesson 2 — ANN architecture: stacking neurons to fix XOR, then extending RR Finance

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Represent non-linear decision boundaries by stacking layers of neurons, each with its own weights, biases and activation function. |
| **2. Why does it matter in finance?** | RR Finance's Module 1 logistic-regression classifier is a *single* neuron. We now rebuild the same default-prediction task with a small **feedforward ANN** (an input layer, one or two **hidden** layers, an output layer) and compare it directly against the Module 1 baseline. |
| **3. Why this technique?** | A feedforward (fully connected / "dense") network is the right first step up from linear models: still tabular-friendly, still fast to train, but able to represent interactions between features that logistic regression cannot without manual feature engineering. |
| **4. What do the parameters mean?** | Layer widths (`16`, `8` neurons below) control *capacity* — how many distinct patterns the hidden layers can represent. More width/depth is not automatically better; it is a hyperparameter to validate, exactly like `alpha` in Module 1's Ridge regression. |
| **5. What is happening mathematically?** | Each layer computes `a = activation(W·x + b)`, and the output of one layer becomes the input to the next — literally the perceptron equation from Lesson 1, repeated and composed. |
| **6. What happens if we change it?** | Too few neurons/layers underfits (can't represent the pattern). Too many, with too little data or no regularisation, overfits — the network memorises the training set instead of generalising. We will see this directly when we compare train vs. test performance. |

### Note on the dataset

This notebook **loads the exact dataset Module 1 saved** (`data/rr_finance_module1_dataset.csv`) and uses the **same 12 features (including `age`), in the same order** as Module 1's baseline (`FEATURES` in that notebook) — not a reduced subset. This matters: it means the ANN below and the Module 1 logistic-regression baseline are trained and evaluated on genuinely identical inputs, which is what makes the head-to-head comparison in Lesson 2 fair rather than approximate. If the Module 1 CSV is not found (e.g. running this notebook standalone), the cell below regenerates an equivalent synthetic table using the same generation logic as Module 1, so this notebook still runs end-to-end on its own.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

DATA_PATH = Path("data/rr_finance_module1_dataset.csv")

# The exact same 11 features, in the exact same order, as Module 1's baseline classifier.
# NOTE: "age" is deliberately excluded from FEATURES -- Module 1 flagged it as a protected
# attribute under fair-lending regulation and kept it out of every model, not just its own baseline.
FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]

if DATA_PATH.exists():
    print("Loading the real Module 1 dataset -- this notebook is continuing that system, not restarting it.")
    df = pd.read_csv(DATA_PATH)
    X = df[FEATURES].to_numpy(dtype=np.float32)
    y = df["default"].to_numpy(dtype=np.float32)
else:
    print("Module 1 CSV not found -- regenerating a comparable synthetic RR Finance table so this")
    print("notebook still runs standalone. Run Module 1 first for the real, validated dataset.")
    rng = np.random.default_rng(SEED)
    n = 800
    age                   = rng.integers(21, 65, size=n)
    annual_income         = rng.lognormal(13.5, 0.5, n)
    monthly_debt          = rng.lognormal(10.0, 0.6, n)
    loan_amount           = rng.lognormal(13.0, 0.6, n)
    loan_term_months      = rng.choice([12, 24, 36, 48, 60, 72], size=n)
    credit_score          = np.clip(rng.normal(680, 80, n), 450, 900).round(0)
    employment_years      = np.clip(rng.exponential(6, n), 0, 30).round(1)
    account_age_months    = np.clip(rng.exponential(60, n), 1, 200).round(0)
    num_previous_loans    = rng.integers(0, 7, size=n)
    previous_defaults     = rng.binomial(num_previous_loans, 0.1)
    debt_to_income = monthly_debt / (annual_income / 12)
    loan_to_income = loan_amount / annual_income

    logit = (-1.0 + 2.5 * debt_to_income + 0.6 * loan_to_income - 0.006 * (credit_score - 650)
             + 0.3 * previous_defaults + rng.normal(0, 0.5, n))
    prob_default = 1 / (1 + np.exp(-logit))
    default = (rng.random(n) < prob_default).astype(np.float32)

    df = pd.DataFrame({
        "age": age, "annual_income": annual_income, "monthly_debt": monthly_debt, "loan_amount": loan_amount,
        "loan_term_months": loan_term_months, "credit_score": credit_score,
        "employment_years": employment_years, "account_age_months": account_age_months,
        "num_previous_loans": num_previous_loans, "previous_defaults": previous_defaults,
        "debt_to_income": debt_to_income, "loan_to_income": loan_to_income, "default": default,
    })
    X = df[FEATURES].to_numpy(dtype=np.float32)
    y = df["default"].to_numpy(dtype=np.float32)

print("Feature matrix shape:", X.shape, " | Default rate:", y.mean().round(3))


In [ ]:
# Same split discipline as Module 1: stratified, scaler fit on TRAIN ONLY
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,   dtype=torch.float32).view(-1, 1)

print("Train tensor:", X_train_t.shape, " Test tensor:", X_test_t.shape)


In [ ]:
class DefaultANN(nn.Module):
    """RR Finance default classifier, v2: replaces the Module 1 logistic-regression baseline."""
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),   # raw logit -- NOT passed through sigmoid here
        )

    def forward(self, x):
        return self.net(x)   # BCEWithLogitsLoss applies the sigmoid internally, for numerical stability

model_ann = DefaultANN(n_features=X_train_t.shape[1])
print(model_ann)

n_params = sum(p.numel() for p in model_ann.parameters())
print("Total trainable parameters:", n_params)


> **Why is the sigmoid missing from the model?** We deliberately leave the final layer as a raw "logit" and use `nn.BCEWithLogitsLoss`, which combines sigmoid + binary cross-entropy in one numerically stable step. This is the standard PyTorch pattern — applying `sigmoid` yourself and then a separate `BCELoss` is mathematically equivalent but less numerically stable. We convert to probabilities only when we need them for evaluation (`torch.sigmoid(logits)`).


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_ann.parameters(), lr=0.01)

train_losses = []
EPOCHS = 150

model_ann.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    logits = model_ann(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | training loss: {loss.item():.4f}")

plt.figure(figsize=(6,3.5))
plt.plot(train_losses)
plt.title("RR Finance ANN v2 -- training loss")
plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.grid(alpha=.3)
plt.show()


In [ ]:
model_ann.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(model_ann(X_test_t)).numpy().ravel()
    test_preds = (test_probs >= 0.5).astype(int)

auc = roc_auc_score(y_test, test_probs)
print(f"RR Finance ANN v2 -- Test ROC-AUC: {auc:.4f}")
print()
print(confusion_matrix(y_test, test_preds))
print()
print(classification_report(y_test, test_preds, target_names=["No default", "Default"]))


### Compare against the Module 1 baseline -- loading the ACTUAL saved model, not a re-derived one

Module 1 saved its trained baseline pipeline to `artifacts/baseline_logreg_pipeline.joblib` after evaluating it on this exact train/test split (same features, same `SEED`, same `stratify`). We load that saved pipeline directly and score it on **this notebook's** `X_test` -- if the split discipline in both notebooks is genuinely identical, this reproduces the exact AUC Module 1 reported, which is the real test of whether "one continuous system" is more than a slogan here. If the artifact is not found (standalone run), we fall back to retraining a logistic regression locally so the comparison still works.

This is now our real, validated RR Finance data (800 rows, ROC-AUC ≈ 0.70 at the Module 1 baseline) -- do not assume the ANN will automatically win -- run the cells and read the actual numbers. The honest lesson stands either way: **do not reach for a neural network before checking whether a simpler model already solves the problem.** That restraint is itself a form of engineering discipline, not a weakness.

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression

BASELINE_PATH = Path("artifacts/baseline_logreg_pipeline.joblib")

if BASELINE_PATH.exists():
    print("Loading Module 1's ACTUAL saved baseline pipeline (scaler + logistic regression).")
    module1_pipeline = joblib.load(BASELINE_PATH)
    logreg_probs = module1_pipeline.predict_proba(X_test)[:, 1]   # pipeline applies its OWN fitted scaler internally
    logreg_auc = roc_auc_score(y_test, logreg_probs)
    source_note = "(loaded from Module 1's saved artifact)"
else:
    print("artifacts/baseline_logreg_pipeline.joblib not found -- retraining a local logistic regression fallback.")
    logreg = LogisticRegression(max_iter=2000)
    logreg.fit(X_train_s, y_train)
    logreg_probs = logreg.predict_proba(X_test_s)[:, 1]
    logreg_auc = roc_auc_score(y_test, logreg_probs)
    source_note = "(retrained locally -- standalone fallback)"

print(f"Module 1 Logistic Regression -- Test ROC-AUC: {logreg_auc:.4f}  {source_note}")
print(f"Module 2 ANN (v2)            -- Test ROC-AUC: {auc:.4f}")
print(f"\\nDifference (ANN - LogReg): {auc - logreg_auc:+.4f}")


---
## Lesson 3 — Activation functions: ReLU, Sigmoid, Softmax

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Without a non-linear activation between layers, stacking any number of linear layers collapses mathematically into a *single* linear layer — depth would buy us nothing. Activations are what give depth its power. |
| **2. Why does it matter in finance?** | The *choice* of activation on the output layer determines what the network's output even means: a probability of default (sigmoid), a probability distribution over 10 cheque digits (softmax), or an unrestricted internal signal (ReLU, hidden layers only). |
| **3. Why these three?** | Sigmoid and Softmax are the standard choices for binary and multi-class outputs respectively. ReLU is today's default for *hidden* layers because it trains faster and avoids a specific failure mode (vanishing gradients) that Sigmoid/Tanh suffer from in deep networks. |
| **4. What do the parameters mean?** | These functions have no tunable parameters themselves — the "parameters" that matter are *where* you place them (hidden vs. output layer) and *which* loss function you pair them with. |
| **5. What is happening mathematically?** | `ReLU(z)=max(0,z)`; `Sigmoid(z)=1/(1+e^-z)` maps to (0,1); `Softmax(z_i)=e^{z_i} / Σ_j e^{z_j}` maps a vector to a probability distribution that sums to 1. |
| **6. What happens if we change it?** | Using Sigmoid in deep hidden layers can shrink gradients toward zero as they propagate backward (vanishing gradients), slowing or stalling learning — this is a large part of why ReLU became the default for hidden layers in the 2010s. |


In [ ]:
z = np.linspace(-6, 6, 400)

relu = np.maximum(0, z)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(z, relu, color="tab:blue"); axes[0].set_title("ReLU (hidden layers)")
axes[1].plot(z, sigmoid, color="tab:orange"); axes[1].set_title("Sigmoid (binary output)")
axes[2].plot(z, tanh, color="tab:green"); axes[2].set_title("Tanh (reference)")
for ax in axes:
    ax.axhline(0, color="grey", lw=.5); ax.axvline(0, color="grey", lw=.5); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Softmax needs a VECTOR of scores, not a single number -- e.g. 10 raw scores, one per cheque digit 0-9
raw_scores = np.array([2.0, 0.5, -1.0, 3.2, 0.1, -0.5, 1.1, 0.0, -2.0, 0.8])

def softmax(scores):
    shifted = scores - np.max(scores)     # subtract max for numerical stability -- does not change the result
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum()

probs = softmax(raw_scores)
print("Raw digit scores:", np.round(raw_scores, 2))
print("Softmax probabilities:", np.round(probs, 3))
print("Sum of probabilities (must be 1.0):", probs.sum().round(6))
print("Predicted digit:", probs.argmax())


> **Trainer note on the softmax stability trick:** subtracting `max(scores)` before exponentiating does **not** change the mathematical result (softmax is shift-invariant), but it prevents `np.exp()` from overflowing on large scores. PyTorch's `nn.Softmax` / `F.log_softmax` do this internally — you rarely need to write it by hand in real code, but you should know *why* it is safe.


---
## Lesson 4 — Forward and backward propagation, by hand

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | How does a network actually learn *which direction* to move each of its (possibly millions of) weights, given only a single scalar loss number at the end? |
| **2. Why does it matter in finance?** | This is the mechanism underneath every `.backward()` call in every model in this course, including the LLMs in Module 4. Trusting a black box you don't understand is a risk in itself — you cannot reason about training instability, exploding losses, or vanishing gradients without this. |
| **3. Why do this by hand once?** | PyTorch's autograd computes this automatically and you will never hand-derive gradients again after this lesson — but doing it once, with a **numerical gradient check**, builds the intuition needed to debug real training runs later. |
| **4. What do the parameters mean?** | We use a tiny 2-input → 3-hidden → 1-output network purely so the arithmetic is checkable by hand/eye. |
| **5. What is happening mathematically?** | **Forward pass:** `z1 = X·W1+b1`, `a1 = σ(z1)`, `z2 = a1·W2+b2`, `a2 = σ(z2)`. **Backward pass (chain rule):** the error at the output is propagated backward layer by layer — `dW2 = a1ᵀ·dZ2`, then `dZ1 = (dZ2·W2ᵀ) ⊙ σ'(a1)`, then `dW1 = Xᵀ·dZ1`. Each gradient answers: "if I nudge this weight up slightly, does the loss go up or down, and by how much?" |
| **6. What happens if we change it?** | This is literally what gradient descent does next (Lesson 5): `W ← W − lr·dW`, repeated until the loss stops improving. |

We verify our hand-derived gradient with a **numerical gradient check** — perturb one weight by a tiny `ε`, measure how much the loss changes, and confirm it matches the analytical (chain-rule) gradient. This check is a standard debugging tool used in real deep-learning engineering, not just a teaching exercise.


In [ ]:
def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(a):
    # derivative of sigmoid, expressed in terms of its OWN output "a" -- a common and efficient trick
    return a * (1 - a)

# Tiny toy dataset: 2 samples, 2 features each
X_toy = np.array([[0.5, -0.2], [0.1, 0.9]])
y_toy = np.array([[1.0], [0.0]])

rng = np.random.default_rng(SEED)
W1 = rng.normal(0, 0.1, size=(2, 3)); b1 = np.zeros((1, 3))
W2 = rng.normal(0, 0.1, size=(3, 1)); b2 = np.zeros((1, 1))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = sigmoid_np(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid_np(z2)
    return z1, a1, z2, a2

def mse_loss(a2, y):
    return np.mean((a2 - y) ** 2)

z1, a1, z2, a2 = forward(X_toy, W1, b1, W2, b2)
loss = mse_loss(a2, y_toy)
print("Forward pass loss:", loss)

# --- Backward pass (manual chain rule) ---
m = X_toy.shape[0]
dA2 = (a2 - y_toy) * (2 / m)
dZ2 = dA2 * sigmoid_deriv(a2)
dW2 = a1.T @ dZ2
db2 = dZ2.sum(axis=0, keepdims=True)

dA1 = dZ2 @ W2.T
dZ1 = dA1 * sigmoid_deriv(a1)
dW1 = X_toy.T @ dZ1
db1 = dZ1.sum(axis=0, keepdims=True)

print("Analytical dW1[0,0]:", dW1[0, 0])

# --- Numerical gradient check on W1[0,0] ---
eps = 1e-5
W1_plus = W1.copy();  W1_plus[0, 0] += eps
W1_minus = W1.copy(); W1_minus[0, 0] -= eps

_, _, _, a2_plus  = forward(X_toy, W1_plus,  b1, W2, b2)
_, _, _, a2_minus = forward(X_toy, W1_minus, b1, W2, b2)

numerical_grad = (mse_loss(a2_plus, y_toy) - mse_loss(a2_minus, y_toy)) / (2 * eps)
print("Numerical  dW1[0,0]:", numerical_grad)
print("Difference:", abs(dW1[0, 0] - numerical_grad), " (should be tiny, e.g. < 1e-6)")


**What just happened, in plain language:** we asked "if `W1[0,0]` were slightly larger, would the loss go up or down, and by how much?" two different ways — once by re-running the *entire* forward pass twice (numerical check, slow, only for verification) and once by the chain-rule shortcut (analytical, fast, what PyTorch's `.backward()` actually does at scale). They agree, which confirms the chain-rule derivation above is correct.

Now watch PyTorch's `autograd` do exactly this, automatically, for a network of any size:


In [ ]:
X_toy_t = torch.tensor(X_toy, dtype=torch.float32)
y_toy_t = torch.tensor(y_toy, dtype=torch.float32)

toy_net = nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1), nn.Sigmoid())
pred = toy_net(X_toy_t)
loss_torch = F.mse_loss(pred, y_toy_t)
loss_torch.backward()   # <-- this single call does everything Lesson 4's manual code did, for a network of ANY size

first_layer_grad = toy_net[0].weight.grad
print("PyTorch autograd computed gradients automatically.")
print("Gradient shape for the first layer's weights:", first_layer_grad.shape)
print(first_layer_grad)


---
## Lesson 5 — Gradient descent and optimizers: SGD, Momentum, Adam, RMSprop

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Given a gradient (Lesson 4 told us *which direction* reduces the loss), how exactly should we step — how far, and should we remember previous steps? |
| **2. Why does it matter in finance?** | A poorly chosen optimiser/learning-rate can make an otherwise-correct model fail to converge, converge to a worse solution, or waste compute budget — all of which have real cost when training RR Finance's production models. |
| **3. Why these four?** | **SGD** is the textbook baseline. **Momentum** adds "memory" of previous steps to smooth out noisy updates. **RMSprop** adapts the step size *per parameter* based on recent gradient magnitude. **Adam** combines both ideas and is the most common default in practice today. |
| **4. What do the parameters mean?** | `lr` (learning rate) is the step size. `momentum` (SGD) controls how much of the previous step's direction carries forward. Adam/RMSprop have internal per-parameter learning-rate adaptation you rarely need to hand-tune beyond `lr` itself. |
| **5. What is happening mathematically?** | **SGD:** `W ← W − lr·∇L`. **Momentum:** `v ← β·v + ∇L; W ← W − lr·v` (a rolling average of gradients). **RMSprop/Adam** additionally track a rolling average of the *squared* gradient to shrink the step size in directions that have been changing rapidly, and enlarge it in directions that have been flat. |
| **6. What happens if we change it?** | Learning rate too high → loss oscillates or diverges. Too low → training crawls. Wrong optimiser for the problem → slower convergence or a worse final minimum. There is no universally "correct" optimiser — validate, exactly as with `alpha` in Module 1's Ridge/Lasso. |

We compare all four on the same tiny regression problem so the differences in convergence speed are visible directly in the loss curves.


In [ ]:
def make_toy_regression(seed=0):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(200, 10, generator=g)
    true_w = torch.randn(10, 1, generator=g)
    y = X @ true_w + 0.1 * torch.randn(200, 1, generator=g)
    return X, y

def make_model():
    torch.manual_seed(0)   # SAME initial weights for every optimizer -- a fair comparison
    return nn.Linear(10, 1)

X_reg, y_reg = make_toy_regression()

optimizers = {
    "SGD":       lambda params: optim.SGD(params, lr=0.05),
    "Momentum":  lambda params: optim.SGD(params, lr=0.05, momentum=0.9),
    "RMSprop":   lambda params: optim.RMSprop(params, lr=0.05),
    "Adam":      lambda params: optim.Adam(params, lr=0.05),
}

loss_histories = {}
for name, make_opt in optimizers.items():
    model = make_model()
    opt = make_opt(model.parameters())
    losses = []
    for step in range(80):
        opt.zero_grad()
        pred = model(X_reg)
        loss = F.mse_loss(pred, y_reg)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    loss_histories[name] = losses
    print(f"{name:9s} -> final loss after 80 steps: {losses[-1]:.5f}")

plt.figure(figsize=(7, 4))
for name, losses in loss_histories.items():
    plt.plot(losses, label=name)
plt.title("Same problem, same starting weights -- optimizer convergence compared")
plt.xlabel("Training step"); plt.ylabel("MSE loss"); plt.legend(); plt.grid(alpha=.3)
plt.show()


> **Trainer question:** Ask participants to change `lr=0.05` to `lr=0.5` for plain SGD and re-run. In most seeds this will oscillate or diverge — a direct, visible demonstration of why "just increase the learning rate to train faster" is not free.


---
## Lesson 6 — Batch normalisation and dropout

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Two different problems that are often taught together because both stabilise training: (a) internal signals drifting to very large/small scales as they pass through layers (solved by **BatchNorm**), and (b) the network memorising the training set instead of generalising (solved by **Dropout**). |
| **2. Why does it matter in finance?** | RR Finance's cheque-digit CNN (Lesson 9) is deep enough that both problems can appear. A model that memorises the training cheques instead of learning generalisable digit shapes will fail in production on cheques it has never seen — a costly, hard-to-detect failure mode. |
| **3. Why these techniques?** | BatchNorm and Dropout are the two most common, well-understood regularisation/stabilisation tools in modern deep learning, and both are one line of PyTorch code per layer. |
| **4. What do the parameters mean?** | `Dropout(p=0.25)` randomly zeroes 25% of a layer's activations *during training only* — forcing the network not to rely too heavily on any single neuron. `BatchNorm2d(channels)` normalises each mini-batch's activations to zero mean / unit variance, per channel, then lets the network re-learn an optimal scale/shift. |
| **5. What is happening mathematically?** | BatchNorm: `x̂ = (x − μ_batch) / √(σ²_batch + ε)`, then `y = γ·x̂ + β` where `γ, β` are learned. Dropout: each activation is independently zeroed with probability `p`, and surviving activations are scaled by `1/(1−p)` so the expected magnitude stays constant. |
| **6. What happens if we change it?** | `p` too high (e.g. 0.8) can under-train the network (too much information destroyed each step); `p=0` disables the regularisation effect entirely. Crucially: **both layers behave differently in training vs. evaluation mode** — this is a very common real-world bug, demonstrated below. |


In [ ]:
layer_dropout = nn.Dropout(p=0.5)
x_demo = torch.ones(1, 10)

layer_dropout.train()   # TRAINING mode: dropout is ACTIVE
print("train() mode  ->", layer_dropout(x_demo))

layer_dropout.eval()    # EVALUATION mode: dropout is a NO-OP (passes values through unchanged)
print("eval() mode   ->", layer_dropout(x_demo))

print()
print("COMMON BUG: forgetting model.eval() before evaluation/inference leaves Dropout and")
print("BatchNorm running in their TRAINING behaviour, silently corrupting predictions.")
print("Always call model.eval() before evaluating or deploying, and model.train() before resuming training.")


---
## Lesson 7 — CNN foundations: convolution, padding, stride, pooling

### The business problem that motivates this lesson

RR Finance starts receiving scanned cheque images with handwritten amounts. Each image is a grid of pixel intensities — flattening a 28×28 image into 784 independent numbers (what a plain ANN would do) throws away the fact that neighbouring pixels are related. A CNN instead slides small, learnable pattern-detectors ("filters" or "kernels") across the image, so the same edge/curve detector can fire wherever that pattern appears — top-left corner or bottom-right, it does not matter.

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Detect local spatial patterns (edges, curves, loops) in an image, regardless of where in the image they appear. |
| **2. Why does it matter in finance?** | This is the mechanism behind cheque-digit recognition, document/table extraction (Module 8), and fraud-image analysis generally. |
| **3. Why convolution?** | A fully-connected layer applied directly to a 28×28 image would need 784 separate weights *per neuron*, with no spatial reuse. A small convolution filter (e.g. 3×3 = 9 weights) is reused at every position in the image — vastly fewer parameters, and the pattern-detection is position-independent by construction. |
| **4. What do the parameters mean?** | **Kernel size** (e.g. 3×3): how large a local neighbourhood each filter looks at. **Padding**: how many zero-pixels we add around the border, so the output doesn't shrink (or edge information isn't lost). **Stride**: how many pixels the filter moves between applications — stride 2 roughly halves the output size. **Pooling**: a downsampling step (e.g. `MaxPool2d(2,2)`) that keeps the strongest signal in each small region and discards the rest, making the representation more compact and slightly position-tolerant. |
| **5. What is happening mathematically?** | At each position, the filter and the underlying image patch are multiplied element-wise and summed (a dot product) to produce one output pixel. Output size formula: `output = floor((input + 2·padding − kernel) / stride) + 1`. |
| **6. What happens if we change it?** | No padding shrinks the output every layer (and edge pixels get seen by fewer filter positions than centre pixels — an "edge effect"). Larger stride throws away spatial resolution faster. Larger kernels see bigger patterns but cost more compute per filter application. |

We implement one convolution **by hand with plain NumPy** first — no PyTorch — so the sliding-window mechanic is completely transparent before we let `nn.Conv2d` do it for us at speed.


In [ ]:
def conv2d_manual(image, kernel, stride=1, padding=0):
    """A from-scratch 2D convolution -- purely to make the sliding-window mechanic visible."""
    if padding > 0:
        image = np.pad(image, ((padding, padding), (padding, padding)))
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = (ih - kh) // stride + 1
    ow = (iw - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = image[i*stride : i*stride+kh, j*stride : j*stride+kw]
            out[i, j] = np.sum(region * kernel)
    return out

# A tiny 5x5 "image"
image = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 1],
    [1, 0, 1, 2, 0],
    [2, 1, 0, 1, 3],
    [0, 2, 1, 0, 1],
], dtype=float)

# A classic edge-detection kernel
edge_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=float)

out_no_pad = conv2d_manual(image, edge_kernel, stride=1, padding=0)
print("No padding, stride 1 -> output shape:", out_no_pad.shape, "(5x5 input shrinks to 3x3)")

out_padded = conv2d_manual(image, edge_kernel, stride=1, padding=1)
print("Padding=1, stride 1  -> output shape:", out_padded.shape, "(same size as input: 'SAME' padding)")

out_strided = conv2d_manual(image, edge_kernel, stride=2, padding=1)
print("Padding=1, stride 2  -> output shape:", out_strided.shape, "(stride downsamples the output)")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(image, cmap="gray"); axes[0].set_title("Input 5x5")
axes[1].imshow(out_padded, cmap="gray"); axes[1].set_title("After edge-kernel convolution\n(padding=1, same size)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# Max pooling: keep the strongest signal in each 2x2 region, discard the rest
def max_pool_manual(feature_map, size=2, stride=2):
    h, w = feature_map.shape
    oh, ow = h // stride, w // stride
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            region = feature_map[i*stride:i*stride+size, j*stride:j*stride+size]
            out[i, j] = region.max()
    return out

pooled = max_pool_manual(out_padded)
print("Feature map before pooling:", out_padded.shape)
print("Feature map after 2x2 max pooling:", pooled.shape, "-- half the spatial size, strongest signals kept")


### Now the PyTorch equivalent (what you will actually use)

`nn.Conv2d` performs exactly the sliding-window operation above — with *learnable* filters instead of a hand-picked edge kernel, and with GPU-accelerated batched computation across many images and many filters at once.


In [ ]:
conv_layer = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1, stride=1)
pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)

# a batch of 4 single-channel 28x28 images (this is exactly MNIST's shape)
batch = torch.randn(4, 1, 28, 28)

after_conv = conv_layer(batch)
after_pool = pool_layer(after_conv)

print("Input batch shape:      ", batch.shape,      " (batch, channels, height, width)")
print("After Conv2d (8 filters):", after_conv.shape, " -- 8 learned feature maps per image, same 28x28 (padding=1)")
print("After MaxPool2d(2,2):    ", after_pool.shape,  " -- spatial size halved to 14x14")


---
## Lesson 8 (Lab) — Build and train a CNN: RR Finance cheque digit recognition

### Business framing

RR Finance's operations team currently keys in handwritten cheque amounts by hand. We prototype an automated digit-recognition CNN using the standard **MNIST** handwritten-digit dataset as a stand-in for real cheque-digit crops — the architecture, training loop, and evaluation approach below transfer directly once real, labelled cheque-digit images are available (the labs in Module 8's "Multimodal AI" build on exactly this pattern for document/table extraction).

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Classify a 28×28 grayscale image of a handwritten digit into one of 10 classes (0–9). |
| **2. Why does it matter in finance?** | Automated, auditable digit recognition reduces manual keying error and cost in cheque processing, and is a building block for broader document-AI pipelines. |
| **3. Why this architecture?** | Two convolution+pooling blocks (Lesson 7) extract increasingly abstract spatial features; BatchNorm (Lesson 6) stabilises training; Dropout (Lesson 6) reduces overfitting; a final fully-connected layer + Softmax (Lesson 3) produces the 10-class probability distribution. |
| **4. What do the parameters mean?** | 16 then 32 filters: the network learns progressively more feature detectors as spatial size shrinks. `Dropout(0.25)`: moderate regularisation, validated empirically rather than assumed. |
| **5. What is happening mathematically?** | Forward pass = repeated convolution → activation → pooling (Lesson 7), then a final linear+softmax classification head; training = the same backprop + Adam optimiser mechanics from Lessons 4–5, applied to convolutional weights instead of a flat weight matrix. |
| **6. What happens if we change it?** | Removing BatchNorm/Dropout on this dataset will still often reach a similar accuracy (MNIST is comparatively easy) — but watch the *gap between training and test accuracy* widen without them; that gap is the overfitting signal Lesson 6 was about. |

### Data

`torchvision.datasets.MNIST` will download the dataset automatically the first time this cell runs (requires internet access on your machine).


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),                      # scales pixel values from [0,255] to [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST's known per-channel mean/std -- centres the data around 0
])

train_dataset = datasets.MNIST(root="data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print("Training images:", len(train_dataset), " | Test images:", len(test_dataset))

# Peek at a few samples
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label)); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
class ChequeDigitCNN(nn.Module):
    """A small CNN for 28x28 grayscale digit images -- the RR Finance cheque-digit reader prototype."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1   = nn.Linear(32 * 7 * 7, 128)   # 28 -> pool -> 14 -> pool -> 7
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # 28x28 -> 14x14, 16 channels
        x = self.pool(F.relu(self.bn2(self.conv2(x))))    # 14x14 -> 7x7,   32 channels
        x = x.view(x.size(0), -1)                          # flatten for the fully-connected head
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)   # raw logits -- CrossEntropyLoss applies softmax internally

cnn = ChequeDigitCNN()
n_params = sum(p.numel() for p in cnn.parameters())
print(cnn)
print("\nTotal trainable parameters:", n_params)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cnn.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)

EPOCHS = 3   # MNIST converges fast; increase for a production-quality model

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

for epoch in range(EPOCHS):
    cnn.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = cnn(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_dataset)
    test_acc = evaluate(cnn, test_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss: {train_loss:.4f} | test accuracy: {test_acc:.4f}")

print("\nFinal cheque-digit CNN test accuracy:", round(evaluate(cnn, test_loader), 4))


### Trainer checkpoint

Before moving on, participants should be able to answer: *"Why does the flattened size before `fc1` equal `32 * 7 * 7`, and not `32 * 28 * 28`?"* — the answer is the two `MaxPool2d(2,2)` halvings from Lesson 7 (`28 → 14 → 7`), tying this lab directly back to the convolution/pooling arithmetic taught two lessons ago.


---
## Lesson 9 — Transfer learning (brief)

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Training a CNN from scratch (Lesson 8) needs a reasonably large labelled dataset. RR Finance may only have a few thousand real, labelled cheque-digit crops — not enough to train a deep CNN from zero reliably. |
| **2. Why does it matter in finance?** | Labelled financial data is expensive and slow to collect (often needs manual annotation under confidentiality constraints). Reusing a network already trained on millions of general images and only fine-tuning it on RR Finance's smaller dataset dramatically reduces the data and compute needed. |
| **3. Why this technique?** | Early convolutional layers tend to learn general-purpose features (edges, textures, simple shapes) that are useful across almost any image task. Only the later, more task-specific layers need retraining. |
| **4. What do the parameters mean?** | "Freezing" a layer (`param.requires_grad = False`) means its weights are held fixed during training — only the unfrozen layers (usually the final classifier head, sometimes a few late convolution blocks) are updated. |
| **5. What is happening mathematically?** | Identical backprop mechanics to Lesson 4 — the only change is that gradients are computed and applied only for the unfrozen subset of parameters. |
| **6. What happens if we change it?** | Freezing too much can prevent the network from adapting to real cheque-image quirks (paper texture, ink variation, scan skew). Freezing too little re-introduces the "not enough data" problem transfer learning was meant to solve. This is itself a hyperparameter to validate on a held-out set. |

The pattern below (load a pretrained backbone, freeze it, replace and train only the final layer) is the one you will reuse whenever RR Finance has a new image task but limited labelled data.


In [ ]:
from torchvision import models

# Load a small pretrained backbone (downloads pretrained ImageNet weights the first time)
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze every existing layer -- we do not want to destroy the general-purpose features it already learned
for param in backbone.parameters():
    param.requires_grad = False

# Replace only the final classification layer to match OUR task: 10 cheque-digit classes instead of 1000 ImageNet classes
n_features_in = backbone.fc.in_features
backbone.fc = nn.Linear(n_features_in, 10)   # this new layer's parameters ARE trainable by default

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in backbone.parameters() if not p.requires_grad)
print(f"Trainable parameters (new head only): {trainable:,}")
print(f"Frozen parameters (pretrained backbone): {frozen:,}")
print("\nTraining loop from here on is IDENTICAL to Lesson 8's -- only backbone.fc.parameters() actually change.")


> **Note:** ResNet-18 expects 3-channel (RGB) 224×224 images by default, so real usage would resize/replicate the single-channel cheque-digit crops accordingly. We stop at the freeze/replace pattern here since the full ResNet fine-tuning loop is a straightforward repeat of Lesson 8's training loop with `optimizer = optim.Adam(backbone.fc.parameters(), lr=1e-3)`.


---
## Lesson 10 (Lab) — Cybersecurity: adversarial examples (FGSM and PGD)

### The security question this module has been building toward

Module 1 asked: *"What if the training data is manipulated?"* (data poisoning). This lesson asks the companion question: *"What if a **clean, correctly trained** model is fed a deliberately crafted input at prediction time?"*

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Craft a tiny, often visually imperceptible perturbation to an input image that causes a correctly-trained CNN to misclassify it with high confidence. |
| **2. Why does it matter in finance?** | If RR Finance's cheque-digit reader (or any production image classifier — document fraud detectors, ID verification, deepfake detectors in Module 11) can be fooled by adversarial pixel noise, an attacker could manipulate an amount or bypass a check without triggering an obvious visual red flag for a human reviewer. |
| **3. Why FGSM and PGD?** | **FGSM (Fast Gradient Sign Method)** is the simplest, fastest attack — one gradient step. **PGD (Projected Gradient Descent)** is FGSM applied iteratively with a "projection" back into an allowed perturbation budget after each step — generally a stronger attack, and the standard *benchmark* attack used to evaluate model robustness. |
| **4. What do the parameters mean?** | `epsilon (ε)`: the maximum allowed perturbation size per pixel — this is the attacker's "budget," small enough to try to stay imperceptible. `alpha (α)` (PGD only): the step size per iteration, smaller than ε so the attack can search within the budget over several steps. `steps` (PGD only): how many iterations of refinement. |
| **5. What is happening mathematically?** | **FGSM:** `x_adv = x + ε · sign(∇_x Loss(model(x), y_true))` — take one step in the *direction that increases the loss the fastest*, using only the *sign* of the gradient (not its magnitude) so every pixel moves by exactly ±ε. **PGD:** repeat a smaller FGSM-like step `α · sign(∇_x Loss)` several times, clamping the total perturbation back within `[-ε, +ε]` after every step (the "projection"). |
| **6. What happens if we change it?** | Larger `ε` → more visible perturbation but higher attack success rate. More PGD `steps` → generally stronger attack but more compute. This is the attacker's trade-off between stealth and effectiveness, and it is exactly the parameter space a red-team exercise (Module 4's Garak lab) would sweep across. |

**Note the key insight:** both attacks reuse the exact backpropagation machinery from Lesson 4 — but instead of computing the gradient of the loss *with respect to the weights* (to update the model), we compute the gradient of the loss *with respect to the input pixels* (to update the image). Same chain rule, different target.


In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    """One-step adversarial attack: move each pixel by +/- epsilon in the direction that increases the loss."""
    images = images.clone().detach().requires_grad_(True)
    logits = model(images)
    loss = criterion(logits, labels)
    model.zero_grad()
    loss.backward()

    perturbation = epsilon * images.grad.sign()
    adv_images = (images + perturbation).clamp(-3, 3).detach()   # clamp to a sane normalized-pixel range
    return adv_images


def pgd_attack(model, images, labels, epsilon, alpha, steps):
    """Iterative adversarial attack: several small FGSM-style steps, projected back within the epsilon budget."""
    original = images.clone().detach()
    adv_images = images.clone().detach()

    for _ in range(steps):
        adv_images.requires_grad_(True)
        logits = model(adv_images)
        loss = criterion(logits, labels)
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            adv_images = adv_images + alpha * adv_images.grad.sign()
            perturbation = torch.clamp(adv_images - original, -epsilon, epsilon)   # <- the "projection" in PGD
            adv_images = (original + perturbation).clamp(-3, 3)
        adv_images = adv_images.detach()

    return adv_images


In [ ]:
# Take one real batch of test digits and attack the CNN we just trained in Lesson 8
cnn.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

clean_preds = cnn(images).argmax(dim=1)
clean_acc = (clean_preds == labels).float().mean().item()
print(f"Clean accuracy on this batch: {clean_acc:.2%}")

for eps in [0.05, 0.15, 0.30]:
    adv_images = fgsm_attack(cnn, images, labels, epsilon=eps)
    adv_preds = cnn(adv_images).argmax(dim=1)
    adv_acc = (adv_preds == labels).float().mean().item()
    print(f"FGSM  epsilon={eps:.2f}  -> accuracy under attack: {adv_acc:.2%}")

adv_images_pgd = pgd_attack(cnn, images, labels, epsilon=0.15, alpha=0.02, steps=10)
pgd_preds = cnn(adv_images_pgd).argmax(dim=1)
pgd_acc = (pgd_preds == labels).float().mean().item()
print(f"\nPGD   epsilon=0.15, 10 steps -> accuracy under attack: {pgd_acc:.2%}")
print("Compare this to FGSM at the same epsilon=0.15 above -- PGD is typically the stronger attack.")


In [ ]:
# Visualise: same digits, before and after a PGD attack -- the perturbation should look like faint noise
eps_demo = 0.15
adv_demo = pgd_attack(cnn, images[:6], labels[:6], epsilon=eps_demo, alpha=0.02, steps=10)
clean_pred_demo = cnn(images[:6]).argmax(dim=1)
adv_pred_demo = cnn(adv_demo).argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for i in range(6):
    axes[0, i].imshow(images[i].detach().cpu().squeeze(), cmap="gray")
    axes[0, i].set_title(f"Clean\npred: {clean_pred_demo[i].item()}")
    axes[0, i].axis("off")

    axes[1, i].imshow(adv_demo[i].detach().cpu().squeeze(), cmap="gray")
    color = "red" if adv_pred_demo[i] != labels[i] else "green"
    axes[1, i].set_title(f"Adversarial (eps={eps_demo})\npred: {adv_pred_demo[i].item()}", color=color)
    axes[1, i].axis("off")

plt.suptitle("Top row: original digits. Bottom row: PGD-perturbed. True label unchanged in both rows.")
plt.tight_layout(); plt.show()


> **Security lesson, stated plainly (same framing as Module 1's poisoning lesson):** the attacker in this scenario needed **no access to training data and no ability to retrain the model.** They only needed (a) the ability to query the model's gradients — a common situation for open, self-hosted, or reverse-engineered models — and (b) a small perturbation budget. This is exactly why adversarial robustness is a distinct security concern from data provenance, and why it gets its own dedicated module later (Module 11: AI Cybersecurity covers black-box/query-only attacks where the attacker cannot even see the gradients directly).


---
## Lesson 11 (Lab) — Adversarial training: a first defence

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Can we make the CNN *inherently* more resistant to the FGSM/PGD attacks from Lesson 10, rather than only detecting attacks after the fact? |
| **2. Why does it matter in finance?** | A more robust cheque-digit reader degrades more gracefully under adversarial or noisy real-world conditions (poor scans, compression artefacts, or deliberate manipulation), reducing both fraud risk and false-alarm operational cost. |
| **3. Why this technique?** | **Adversarial training** is the most direct and widely used defence: generate adversarial examples *during* training and include them in the training loss, so the model is explicitly taught to classify them correctly too — not just clean images. |
| **4. What do the parameters mean?** | The mixing weight between clean-image loss and adversarial-image loss (`0.5 / 0.5` below) controls the trade-off between clean accuracy and robust accuracy — pushing too far toward adversarial loss can measurably reduce clean-data accuracy, a well-documented trade-off in the adversarial ML literature. |
| **5. What is happening mathematically?** | Standard training minimises `Loss(model(x), y)`. Adversarial training minimises a blend: `0.5·Loss(model(x), y) + 0.5·Loss(model(FGSM(x)), y)` — the model now receives gradient signal from *both* the clean image and an attacked version of it, every step. |
| **6. What happens if we change it?** | Training against FGSM only provides some protection against FGSM, but a model trained with FGSM adversarial training is not automatically robust to PGD (a stronger attack) — this is measured directly in the evaluation cell below, and it is the reason robustness claims must always specify *which* attack, and at *which* epsilon, was tested. |

We take a **fresh copy** of the Lesson 8 architecture, retrain it with adversarial examples mixed in, then compare its robustness against the *original* (non-adversarially-trained) CNN under the same PGD attack from Lesson 10.


In [ ]:
robust_cnn = ChequeDigitCNN().to(device)
robust_optimizer = optim.Adam(robust_cnn.parameters(), lr=1e-3)

ADV_EPOCHS = 3
ADV_EPSILON = 0.15

for epoch in range(ADV_EPOCHS):
    robust_cnn.train()
    running_loss = 0.0
    for images_b, labels_b in train_loader:
        images_b, labels_b = images_b.to(device), labels_b.to(device)

        # generate an FGSM adversarial version of this batch, using the model's CURRENT weights
        adv_images_b = fgsm_attack(robust_cnn, images_b, labels_b, epsilon=ADV_EPSILON)

        robust_optimizer.zero_grad()
        clean_logits = robust_cnn(images_b)
        adv_logits   = robust_cnn(adv_images_b)
        loss = 0.5 * criterion(clean_logits, labels_b) + 0.5 * criterion(adv_logits, labels_b)
        loss.backward()
        robust_optimizer.step()
        running_loss += loss.item() * images_b.size(0)

    clean_acc_epoch = evaluate(robust_cnn, test_loader)
    print(f"Adv-train epoch {epoch+1}/{ADV_EPOCHS} | mixed loss: {running_loss/len(train_dataset):.4f} "
          f"| clean test accuracy: {clean_acc_epoch:.4f}")


In [ ]:
# Head-to-head robustness comparison: ORIGINAL (Lesson 8) CNN vs. ADVERSARIALLY-TRAINED (Lesson 11) CNN
cnn.eval(); robust_cnn.eval()

images_eval, labels_eval = next(iter(test_loader))
images_eval, labels_eval = images_eval.to(device), labels_eval.to(device)

def accuracy(model, imgs, lbls):
    with torch.no_grad():
        preds = model(imgs).argmax(dim=1)
    return (preds == lbls).float().mean().item()

print(f"{'Condition':35s} {'Original CNN':>15s} {'Adv-trained CNN':>18s}")
print("-" * 70)

clean_orig = accuracy(cnn, images_eval, labels_eval)
clean_robust = accuracy(robust_cnn, images_eval, labels_eval)
print(f"{'Clean test images':35s} {clean_orig:>14.2%} {clean_robust:>17.2%}")

for eps in [0.05, 0.15, 0.30]:
    adv_for_orig   = fgsm_attack(cnn, images_eval, labels_eval, epsilon=eps)
    adv_for_robust = fgsm_attack(robust_cnn, images_eval, labels_eval, epsilon=eps)
    acc_orig   = accuracy(cnn, adv_for_orig, labels_eval)
    acc_robust = accuracy(robust_cnn, adv_for_robust, labels_eval)
    print(f"{'FGSM attack, eps='+str(eps):35s} {acc_orig:>14.2%} {acc_robust:>17.2%}")

adv_pgd_orig   = pgd_attack(cnn, images_eval, labels_eval, epsilon=0.15, alpha=0.02, steps=10)
adv_pgd_robust = pgd_attack(robust_cnn, images_eval, labels_eval, epsilon=0.15, alpha=0.02, steps=10)
print(f"{'PGD attack, eps=0.15, 10 steps':35s} "
      f"{accuracy(cnn, adv_pgd_orig, labels_eval):>14.2%} "
      f"{accuracy(robust_cnn, adv_pgd_robust, labels_eval):>17.2%}")


### Reading this table honestly

Expect the adversarially-trained model to hold up noticeably better under FGSM at the epsilon it was trained on (`0.15`) than the original model does — that is adversarial training working as intended. Also expect the improvement under **PGD** to be smaller than under FGSM, and possibly modest at higher epsilon: **a model trained against one attack is not automatically robust against a different or stronger attack.** This gap is not a flaw in the demonstration — it is the real, well-documented state of adversarial robustness research, and it is precisely why Module 11 (AI Cybersecurity) treats "the model was adversarially trained" as one layer of defence, not a solved problem, in the same way Module 1 treated Isolation Forest as one layer of a poisoning defence rather than a complete solution.


---
## Module 2 hand-off: what RR Finance now has

| Artifact | What it is | Extends |
|---|---|---|
| `DefaultANN` (Lesson 2) | A multi-layer ANN default classifier, benchmarked directly against the Module 1 logistic-regression baseline | Module 1's default-prediction pipeline |
| `ChequeDigitCNN` (Lesson 8) | A trained CNN for handwritten digit recognition | New modality: cheque-image processing |
| Transfer-learning pattern (Lesson 9) | A freeze-and-fine-tune recipe for future image tasks with limited labelled data | Reused directly in Module 8 (Multimodal AI) |
| `fgsm_attack` / `pgd_attack` (Lesson 10) | Working adversarial-example generators, tested against the trained CNN | Module 1's "attack the system" pattern, now at inference time instead of training time |
| `robust_cnn` (Lesson 11) | An adversarially-trained CNN, with a measured (not assumed) robustness comparison against the original | The first layered AI-security control of this module |

### What Module 3 builds on this

Module 3 (NLP + Financial Text AI) reuses the exact same skeleton one more time: business problem → representation (embeddings, this time, instead of pixels) → model → cybersecurity angle (prompt injection and PII exposure, this time, instead of adversarial pixels) → lab. If Lessons 1–11 of this notebook felt solid, Module 3 will feel like a continuation, not a restart.

### Before Day 2 starts

Run this entire notebook top to bottom once, uninterrupted, to confirm the environment (PyTorch + torchvision + internet access for the MNIST/ResNet downloads) works end-to-end on the delivery machine — not just cell-by-cell during prep.
